# 02 · Collect human texts (Wikipedia, revision as of 2022-11-01)
**Purpose:** for each of the 6 languages, sample random articles and extract the **lead paragraph of the article revision that existed on the cutoff date** (pre-ChatGPT ⇒ very unlikely to be LLM-written).

**Why the API (not dumps):** full dumps for en/ru are tens of GB; the MediaWiki API gives the exact historical revision (`rvstart` = cutoff) and its Wikidata QID (used later for leakage-free cross-lingual splits).

**Resumable:** every API result is appended to `01_raw/wikipedia/{lang}_pages.jsonl`; re-running skips finished titles.
**Outputs:** `01_raw/wikipedia/{lang}_candidates.json` (sampled titles = reproducibility artefact), `{lang}_pages.jsonl`, `02_validated/human_{lang}.parquet`.

**v1.1 (2026-10-10):** after the first audit — consecutive lead paragraphs are joined until `min_words` is reached, bot-template articles are removed, and languages below target are topped up with extra random articles pre-filtered to non-stubs (≥ `wiki_min_bytes`). Pages already fetched are reused (no repeated API calls).

**Runtime:** CPU; first run ~1–2 h; top-up ~10–30 min per language (network-bound).

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "02_collect_human_wikipedia"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
!pip -q install fasttext beautifulsoup4

In [ ]:
import requests, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup

migrate_config(); D = CFG["data"]
# Wikimedia User-Agent policy: identify the tool AND give a contact (URL or e-mail). Edit CONTACT if you prefer another one.
CONTACT = "https://sdu.edu.kz"
UA_STRING = f"MATD-research/1.0 ({CONTACT}; academic Turkic NLP research) python-requests/{requests.__version__}"
SESSION = requests.Session(); SESSION.headers.update({"User-Agent": UA_STRING, "Accept-Encoding": "gzip"})
PAUSE = 0.1          # polite delay after every successful call (per worker)
log.info(f"User-Agent: {UA_STRING}")

def api(lang, params, retries=8):
    """MediaWiki API call with Retry-After/maxlag handling. On failure the error says exactly what the server returned."""
    url = f"https://{LANG_META[lang]['wiki']}.wikipedia.org/w/api.php"
    params = {"format": "json", "formatversion": 2, "maxlag": 5, **params}
    last = None
    for a in range(retries):
        try:
            r = SESSION.get(url, params=params, timeout=30)
        except requests.RequestException as e:
            last = f"network error: {e!r}"; time.sleep(min(60, 2 ** a)); continue
        if r.status_code in (429, 503):                       # rate limited / overloaded
            last = f"HTTP {r.status_code}: {r.text[:200]}"
            time.sleep(min(120, float(r.headers.get("Retry-After", 2 ** a)))); continue
        if r.status_code != 200:
            last = f"HTTP {r.status_code}: {r.text[:300]}"
            if r.status_code == 403:                            # policy block: retrying will not help
                break
            time.sleep(min(60, 2 ** a)); continue
        try:
            j = r.json()
        except ValueError:
            last = f"non-JSON response: {r.text[:300]}"; time.sleep(min(60, 2 ** a)); continue
        err = j.get("error")
        if err:
            if err.get("code") == "maxlag":                     # servers lagging: wait as instructed
                last = "maxlag"; time.sleep(float(r.headers.get("Retry-After", 5))); continue
            raise RuntimeError(f"API error [{lang}] {err.get('code')}: {err.get('info')}")
        time.sleep(PAUSE)
        return j
    raise RuntimeError(f"API failed after {retries} tries [{lang}] {params} -> last response: {last}")

### Connectivity check (prints the raw server answer — paste it to the chat if anything below fails)

In [ ]:
for lang in LANGS:
    r = SESSION.get(f"https://{LANG_META[lang]['wiki']}.wikipedia.org/w/api.php", timeout=30,
                    params={"action": "query", "list": "random", "rnnamespace": 0, "rnlimit": 10, "format": "json", "formatversion": 2})
    print(lang, r.status_code, r.text[:160].replace("\n", " "))

## 1 · Sample candidate titles (saved once → reproducible)

In [ ]:
def sample_titles(lang, n):
    path = P["raw_wiki"] / f"{lang}_candidates.json"
    if path.exists():
        return json.load(open(path))["titles"]
    titles = {}                                   # dict keeps random arrival order (no alphabetical bias)
    while len(titles) < n:
        j = api(lang, {"action": "query", "list": "random", "rnnamespace": 0, "rnlimit": 10})   # API max for non-bots is 10
        for x in j["query"]["random"]:
            titles.setdefault(x["title"], None)
    titles = list(titles)[:n]
    write_json(path, {"lang": lang, "n": len(titles), "sampled_utc": utcnow(), "titles": titles})
    return titles

## 2 · Fetch the revision at the cutoff and extract lead paragraphs
Paragraph text is taken with `get_text("")` (no separator) and footnote/reference markup is removed **before** extraction — this avoids the double-space / space-before-punctuation artefacts we found in the COLING Kazakh human texts (a trivial rule on those artefacts reaches 88% accuracy there).

In [ ]:
DROP_SELECTORS = ["sup", "style", "script", ".reference", ".mw-ref", ".noprint", ".mw-empty-elt",
                  ".geo-inline", "#coordinates", ".IPA", ".rt-commentedText", "span[style*='display:none']"]

def lead_paragraphs(html):
    soup = BeautifulSoup(html, "html.parser")
    root = soup.select_one(".mw-parser-output") or soup
    for sel in DROP_SELECTORS:
        for el in root.select(sel):
            el.decompose()
    paras = []
    for el in root.children:
        name = getattr(el, "name", None)
        if name in ("h2",) or (name == "div" and "mw-heading" in (el.get("class") or [])):
            break                                     # lead section ends at the first section heading
        if name == "p":
            t = el.get_text("").strip()
            if t:
                paras.append(t)
    return paras

def fetch_page(lang, title):
    j = api(lang, {"action": "query", "titles": title, "redirects": 1, "prop": "revisions|pageprops",
                   "rvlimit": 1, "rvstart": D["wiki_cutoff_utc"], "rvdir": "older", "rvprop": "ids|timestamp",
                   "ppprop": "wikibase_item|disambiguation"})
    pg = j["query"]["pages"][0]
    rec = {"lang": lang, "title": title, "pageid": pg.get("pageid"), "status": "ok"}
    if pg.get("missing") or not pg.get("revisions"):
        rec["status"] = "no_revision_before_cutoff"; return rec
    pp = pg.get("pageprops", {})
    rec.update(revid=pg["revisions"][0]["revid"], rev_ts=pg["revisions"][0]["timestamp"],
               qid=pp.get("wikibase_item"), disambiguation="disambiguation" in pp)
    if rec["disambiguation"]:
        rec["status"] = "disambiguation"; return rec
    h = api(lang, {"action": "parse", "oldid": rec["revid"], "prop": "text", "disablelimitreport": 1,
                   "disableeditsection": 1})
    rec["paragraphs"] = lead_paragraphs(h["parse"]["text"])
    return rec

def collect(lang, titles, workers=3):
    out = P["raw_wiki"] / f"{lang}_pages.jsonl"
    done = {r["title"] for r in read_jsonl(out)}
    todo = [t for t in titles if t not in done]
    log.info(f"[{lang}] {len(done)} done, {len(todo)} to fetch")
    buf = []
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(fetch_page, lang, t): t for t in todo}
        for i, f in enumerate(as_completed(futs), 1):
            try:
                buf.append(f.result())
            except Exception as e:
                buf.append({"lang": lang, "title": futs[f], "status": f"error: {e}"})
            if len(buf) >= 100 or i == len(todo):
                append_jsonl(out, buf); buf = []
                log.info(f"[{lang}] {i}/{len(todo)}")
    return read_jsonl(out)

In [ ]:
for lang in LANGS:
    titles = sample_titles(lang, D["n_candidates_per_lang"])
    collect(lang, titles)

## 1b · Extra candidates for top-up (random articles pre-filtered to non-stubs)
`generator=random` + `prop=info` returns each page's current size in bytes in the same call, so stubs are skipped **before** the expensive revision/parse calls. The filter is recorded in the candidates file.

In [ ]:
def sample_extra(lang, n_new):
    path = P["raw_wiki"] / f"{lang}_candidates_extra.json"
    prev = json.load(open(path))["titles"] if path.exists() else []
    known = set(sample_titles(lang, D["n_candidates_per_lang"])) | set(prev)
    # page ids grow with creation time: pages with an id above the largest id seen with a pre-cutoff revision were
    # (almost surely) created after the cutoff, so they are skipped (in uz ~54% of random pages are post-2022)
    pid_max = max([r["pageid"] for r in read_jsonl(P["raw_wiki"] / f"{lang}_pages.jsonl") if r.get("status") == "ok" and r.get("pageid")] or [10**12])
    new, calls = {}, 0
    while len(new) < n_new and calls < 3 * n_new:
        j = api(lang, {"action": "query", "generator": "random", "grnnamespace": 0, "grnlimit": 10, "prop": "info"})
        calls += 1
        for pg in j.get("query", {}).get("pages", []):
            if pg.get("length", 0) >= D["wiki_min_bytes"] and pg.get("pageid", 0) <= pid_max and pg["title"] not in known:
                new.setdefault(pg["title"], None)
    titles = prev + list(new)
    write_json(path, {"lang": lang, "n": len(titles), "updated_utc": utcnow(), "api_calls_last_round": calls,
                      "filter": f"current page length >= {D['wiki_min_bytes']} bytes and pageid <= {pid_max}", "titles": titles})
    log.info(f"[{lang}] extra candidates: +{len(new)} ({calls} random calls)")
    return list(new)

## 3 · Validate: build one text per article, normalise, filter
Rules (identical for all languages):
1. consecutive lead paragraphs are joined until ≥ `min_words` words (after normalisation), then truncated to ≤ `max_words` at a sentence boundary;
2. **bot-template filter:** a text is dropped if > `template_max_share` of its word 4-grams occur in ≥ `template_df` other texts of the same language (catches mass-generated stub templates);
3. GlotLID label must equal the expected language (p ≥ 0.5);
4. Kazakh topics present in the COLING-kk external set are excluded (exact title); near-duplicates are removed in 04.

In [ ]:
from collections import Counter
coling = pd.DataFrame(read_jsonl(P["raw_external"] / "coling2025_kazakh.jsonl"))
coling_topics = set(coling.prompt.str.extract(r"topic(?: in Kazakh)?: (.*?)(?:\.\s|\.$|$)")[0].dropna().str.strip())
log.info(f"COLING-kk distinct topics to exclude: {len(coling_topics)}")

def build_human(lang):
    recs = {}
    for r in read_jsonl(P["raw_wiki"] / f"{lang}_pages.jsonl"):
        if r.get("status") == "ok":
            recs[r["title"]] = r
    rows = []
    for r in recs.values():
        used = []
        for para in r.get("paragraphs", []):
            used.append(para)
            t = normalize_text("\n".join(used))
            if n_words(t) >= D["min_words"]:
                t = truncate_words(t, D["max_words"])
                if n_words(t) >= D["min_words"]:
                    rows.append({k: r.get(k) for k in ("lang", "title", "pageid", "revid", "rev_ts", "qid")} |
                                {"text_raw": "\n".join(used), "n_paragraphs": len(used), "text": t, "n_words": n_words(t)})
                break
    df = pd.DataFrame(rows)
    st = {"lang": lang, "pages_ok": len(recs), "with_text": len(df)}
    if df.empty:
        return df, st
    if lang == "kk":
        df = df[~df.title.isin(coling_topics)].reset_index(drop=True)
    grams = [set(zip(*[w[i:] for i in range(4)])) for w in df.text.str.lower().str.split()]
    dfreq = Counter(x for gs in grams for x in gs)
    df["template_share"] = [sum(dfreq[x] >= D["template_df"] for x in gs) / len(gs) if gs else 0.0 for gs in grams]
    tmpl = df.template_share > D["template_max_share"]
    st["template_dropped"] = int(tmpl.sum()); df = df[~tmpl].reset_index(drop=True)
    lid = langid(df.text.tolist())
    df["lid"], df["lid_prob"] = [l for l, _ in lid], [p for _, p in lid]
    ok = (df.lid == LANG_META[lang]["lid"]) & (df.lid_prob >= D["langid_min_prob"])
    st.update(lid_rejected=int((~ok).sum()), final=int(ok.sum()), joined_paragraphs=int((df[ok].n_paragraphs > 1).sum()),
              median_words=float(df[ok].n_words.median()))
    return df[ok].reset_index(drop=True), st

## 4 · Top-up languages below target, then save
Target per language = `n_topics_target × gen_oversample` validated texts (headroom for generation failures, de-duplication and length matching in 04).

In [ ]:
TARGET = int(np.ceil(D["n_topics_target"] * D["gen_oversample"]))
stats = []
for lang in LANGS:
    df, st = build_human(lang)
    log.info(f"[{lang}] validated {len(df)} / target {TARGET}")
    for rnd in range(D["max_topup_rounds"]):
        if len(df) >= TARGET:
            break
        extra_path = P["raw_wiki"] / f"{lang}_candidates_extra.json"
        extra_all = json.load(open(extra_path))["titles"] if extra_path.exists() else []
        done_extra = {r["title"] for r in read_jsonl(P["raw_wiki"] / f"{lang}_pages.jsonl")} & set(extra_all)
        if extra_all and len(done_extra) < len(extra_all):            # resume an interrupted round first
            new = [t for t in extra_all if t not in done_extra]
        else:
            yield_extra = 0.4                                            # prior: share of pre-filtered pages that validate
            new = sample_extra(lang, min(4000, int(np.ceil((TARGET - len(df)) / yield_extra * 1.1))))
        collect(lang, new)
        df, st = build_human(lang)
        log.info(f"[{lang}] round {rnd + 1}: validated {len(df)} / target {TARGET}")
    st["n_candidates_total"] = len(sample_titles(lang, D["n_candidates_per_lang"])) + \
        (len(json.load(open(P["raw_wiki"] / f"{lang}_candidates_extra.json"))["titles"]) if (P["raw_wiki"] / f"{lang}_candidates_extra.json").exists() else 0)
    st["reached_target"] = len(df) >= TARGET
    stats.append(st); log.info(st)
    save_parquet(df, P["validated"] / f"human_{lang}.parquet")

stats = pd.DataFrame(stats); stats.to_csv(P["validated"] / "human_collection_stats.csv", index=False)
stats

**Check before continuing:** `reached_target` should be True for every language. If not, re-run this cell (it continues with new pre-filtered candidates), or accept a smaller balanced N — notebook 04 balances all cells to the minimum and logs a warning.